<img src = "https://drive.google.com/uc?export=view&id=1VV2e_u46fNm_ewns8QW2HGRZAPHh-e2t" alt = "Encabezado MLDS" width = "100%">  </img>

# Clasificación visual de residuos con aprendizaje profundo
## Fase 3 · Diseño e implementación

**Proyecto aplicado · Módulo 5 · Machine Learning & Data Science**
**Universidad Nacional de Colombia**

| Integrante | Nombre | Rol en esta fase |
|---|---|---|
| A | Juan Carlos Andrade Unigarro | Partición, controles automáticos y trazabilidad |
| B | Natalia Patricia Remolina Rodriguez | Lidera: diseño e implementación de los modelos |

## Resumen ejecutivo
---

| Aspecto | Decisión |
|---|---|
| **Problema** | Clasificación supervisada **multiclase** de una fotografía en 6 materiales (`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) |
| **Partición** | La de la fase 2, fijada archivo por archivo: **1.798 / 360 / 363** imágenes (entrenamiento / validación / prueba), estratificada y sin casi duplicados entre particiones |
| **Pregunta experimental** | ¿La transferencia de aprendizaje desde ImageNet supera a una CNN entrenada desde cero con los mismos datos, la misma entrada y el mismo protocolo? |
| **Modelo A** | CNN pequeña desde cero: 4 bloques Conv–BatchNorm–ReLU–MaxPool, *global average pooling*, *dropout* y salida *softmax* |
| **Modelo B** | **MobileNetV2** preentrenada en ImageNet y congelada, con una cabeza nueva (*global average pooling*, *dropout* y *softmax*); ajuste fino de las últimas capas como variante B-FT |
| **Métrica principal** | **F1 macro en validación**, que pesa igual a las 6 clases. Además: recall por clase, accuracy, matriz de confusión, tiempo por época y número de parámetros |
| **Protocolo** | Entrenamiento ajusta pesos; validación decide parada temprana, hiperparámetros y modelo; **prueba queda reservada para la fase 5** |
| **Desbalance** | Pesos por clase calculados con entrenamiento (`trash`: 3,06) |
| **Referencias a superar** | Mayoritaria F1 0,062 · solo fondo 0,403 · descriptores globales 0,528 (fase 2) · MobileNetV2 congelada + regresión logística (sección 4.5) |
| **Implementación** | Ambos modelos construidos, compilados y verificados: forma de salida, pérdida inicial, sobreajuste de un lote, ensayo corto con medición de tiempo |

## Contenido
---

0. **Entorno, datos y continuidad con las fases 1 y 2**
1. **Partición del conjunto de datos**
2. **Diseño de la configuración experimental**
3. **Selección y diseño de los modelos**
4. **Implementación de los modelos:** pipeline · construcción · compilación · verificaciones · ensayo corto · línea base de transferencia
5. **Controles automáticos y trazabilidad**
6. **Conclusiones y próximos pasos**
7. **Referencias**

## 0. Entorno, datos y continuidad con las fases 1 y 2
---

Esta fase reutiliza sin cambios las decisiones validadas en las fases anteriores:

1. **Fuente:** TrashNet, versión reducida (`dataset-resized.zip`), revisión fijada y verificada con SHA-256.
2. **Exclusiones:** 6 archivos con píxeles idénticos y etiquetas distintas (fase 1); quedan **2.521 imágenes utilizables**.
3. **Partición:** la de la fase 2, que corrigió los casi duplicados entre particiones (26 imágenes pasaron a entrenamiento). Se fija archivo por archivo para que no dependa de la versión de scikit-learn.
4. **Preprocesamiento:** reducción a 160 × 120 y relleno a 160 × 160, caché en memoria, pipeline `tf.data` con aumentos solo en entrenamiento y escala propia de cada modelo.

Se recomienda ejecutar el notebook con **GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU). También funciona en CPU, con un ensayo corto más lento.

In [ ]:
# @title 0.1 Entorno: librerías, semilla y hardware
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import shutil
import time
import urllib.request
import zipfile

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import display, Markdown

import tensorflow as tf
import keras

SEED = 42
keras.utils.set_random_seed(SEED)  # semillas de Python, NumPy y TensorFlow

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
LABEL_ES = dict(zip(CLASSES, ["Cartón", "Vidrio", "Metal", "Papel", "Plástico", "Residual"]))
EXPECTED = {"cardboard": 403, "glass": 501, "metal": 410, "paper": 594, "plastic": 482, "trash": 137}
SPLITS = ["entrenamiento", "validacion", "prueba"]
SPLIT_ES = {"entrenamiento": "Entrenamiento", "validacion": "Validación", "prueba": "Prueba"}

ROOT = Path.cwd() / "residuos_fase3"
ARCHIVE = ROOT / "dataset-resized.zip"
DATA_DIR = ROOT / "imagenes"
REPORT_DIR = ROOT / "reportes"
MODEL_DIR = ROOT / "modelos"
for folder in (ROOT, DATA_DIR, REPORT_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CLASS_COLORS = dict(zip(CLASSES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
SEQUENTIAL = LinearSegmentedColormap.from_list("secuencial", ["#f4f8fd", "#86b6ef", "#104281"])
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9.5, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})


def miles(n):
    """Entero con punto como separador de miles."""
    return f"{int(n):,}".replace(",", ".")


def dec(x, nd=2):
    """Número con coma decimal."""
    return f"{x:,.{nd}f}".replace(",", "#").replace(".", ",").replace("#", ".")


def pct(x, nd=1):
    """Proporción (0-1) como porcentaje con coma decimal."""
    return dec(100 * x, nd) + " %"


GPUS = tf.config.list_physical_devices("GPU")
DEVICE = "GPU" if GPUS else "CPU"
versions = {p: metadata.version(p) for p in ["numpy", "pandas", "Pillow", "scikit-learn", "matplotlib"]}
versions["tensorflow"] = tf.__version__
versions["keras"] = keras.__version__
print("Python:", platform.python_version(), "| Semilla:", SEED, "| Núcleos de CPU:", os.cpu_count())
print("Acelerador:", [g.name for g in GPUS] if GPUS else "ninguno (se usa CPU)")
display(pd.Series(versions, name="versión").to_frame())

In [ ]:
# @title 0.2 Descarga verificada (Hugging Face, con respaldo en GitHub)
HF_REVISION = "94cd17ebfd6702bf62281d3c89f22ff649815b46"
GH_COMMIT = "6fa2b878c6c1b4304b91109070ce0edf9279bb31"
SOURCES = [
    {"nombre": "Hugging Face · revisión fijada en la fase 1",
     "url": f"https://huggingface.co/datasets/garythung/trashnet/resolve/{HF_REVISION}/dataset-resized.zip",
     "sha256": "c060e8abfe5d6de0578ca15be1ed8ad0794a865d333c3473d53d1d9ad6e38b8c"},
    {"nombre": "GitHub del autor · commit fijado (respaldo)",
     "url": f"https://github.com/garythung/trashnet/raw/{GH_COMMIT}/data/dataset-resized.zip",
     "sha256": "0bf472790f8b20e5c950d5b5012a9d38af0d3392efd65f8ce171334fc16b07c2"},
]


def sha256_file(path):
    """Hash por bloques para mantener bajo el uso de memoria."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def obtain_archive(sources, destination):
    """Reutiliza una caché verificada o descarga de la primera fuente disponible."""
    if destination.exists():
        digest = sha256_file(destination)
        for source in sources:
            if digest == source["sha256"]:
                return source, "caché verificada"
    errors = []
    for source in sources:
        temporary = destination.with_suffix(".download")
        try:
            with urllib.request.urlopen(source["url"], timeout=60) as response, temporary.open("wb") as target:
                shutil.copyfileobj(response, target, length=1024 * 1024)
            if sha256_file(temporary) != source["sha256"]:
                raise ValueError("el SHA-256 no coincide con el fijado")
            temporary.replace(destination)
            return source, "descarga verificada"
        except Exception as exc:
            temporary.unlink(missing_ok=True)
            errors.append(f"{source['nombre']}: {type(exc).__name__}")
    raise RuntimeError("No se obtuvo un ZIP válido. " + " | ".join(errors))


t0 = time.perf_counter()
source, acquisition_status = obtain_archive(SOURCES, ARCHIVE)
manifest_names = []
with zipfile.ZipFile(ARCHIVE) as archive:
    for member in archive.infolist():
        parts = Path(member.filename).parts
        if (len(parts) == 3 and parts[0] == "dataset-resized"
                and parts[1] in CLASSES and parts[2].lower().endswith(".jpg")):
            relative = Path(parts[1]) / parts[2]
            target = DATA_DIR / relative
            target.parent.mkdir(exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
            manifest_names.append(relative.as_posix())
manifest_names = sorted(manifest_names)
print("Fuente:", source["nombre"], "|", acquisition_status)
print("Imágenes extraídas:", len(manifest_names), f"| {time.perf_counter() - t0:.1f} s")

In [ ]:
# @title 0.3 Partición fijada en la fase 2 { display-mode: "form" }
# Archivos excluidos en la fase 1: pares con píxeles idénticos y etiquetas distintas.
EXCLUIDAS = ["glass/glass115.jpg", "metal/metal91.jpg", "glass/glass176.jpg",
             "plastic/plastic152.jpg", "glass/glass389.jpg", "plastic/plastic332.jpg"]

# Validación y prueba definitivas de la fase 2 (número de archivo por clase); el resto es entrenamiento.
PARTICION_FASE2 = {
    "validacion": {
        "cardboard": [
            1, 7, 19, 22, 28, 38, 39, 46, 47, 68, 73, 93, 95, 96, 100, 101, 102, 104, 121, 122,
            140, 145, 146, 164, 175, 176, 177, 189, 191, 192, 193, 195, 214, 234, 235, 238,
            255, 257, 259, 260, 265, 274, 285, 286, 301, 302, 303, 320, 327, 329, 340, 347,
            348, 359, 366, 369, 380, 389, 390, 393, 399
        ],
        "glass": [
            41, 43, 51, 52, 60, 63, 65, 66, 73, 82, 97, 114, 116, 118, 126, 131, 133, 134, 137,
            140, 157, 158, 164, 165, 182, 183, 200, 204, 206, 223, 225, 226, 227, 231, 239,
            245, 247, 258, 270, 271, 294, 296, 310, 315, 317, 318, 324, 332, 333, 336, 337,
            356, 362, 365, 366, 384, 385, 386, 409, 410, 420, 421, 424, 426, 430, 442, 452,
            453, 472, 473, 475, 495, 496, 500
        ],
        "metal": [
            10, 44, 45, 46, 47, 49, 70, 74, 87, 92, 114, 115, 116, 124, 134, 136, 138, 144,
            150, 153, 156, 157, 182, 187, 188, 203, 206, 207, 229, 230, 232, 233, 237, 239,
            250, 252, 264, 277, 298, 300, 301, 316, 320, 321, 322, 323, 325, 341, 347, 349,
            370, 374, 375, 391, 392, 410
        ],
        "paper": [
            3, 14, 24, 30, 46, 61, 69, 77, 78, 80, 98, 113, 114, 116, 134, 150, 151, 152, 160,
            161, 165, 189, 190, 192, 207, 214, 216, 224, 230, 232, 249, 252, 255, 257, 259,
            272, 286, 302, 309, 321, 323, 328, 331, 334, 342, 343, 346, 365, 366, 368, 391,
            392, 393, 411, 415, 416, 433, 435, 440, 444, 450, 454, 456, 463, 483, 484, 485,
            503, 504, 505, 525, 529, 530, 531, 535, 539, 556, 558, 563, 585, 590, 591, 592
        ],
        "plastic": [
            27, 28, 30, 32, 34, 36, 43, 48, 52, 53, 62, 65, 69, 84, 90, 91, 100, 104, 112, 124,
            140, 148, 149, 167, 170, 171, 172, 177, 190, 192, 194, 206, 209, 216, 232, 233,
            235, 253, 254, 255, 269, 273, 279, 280, 300, 302, 321, 323, 341, 348, 362, 369,
            383, 385, 401, 402, 403, 404, 407, 412, 433, 448, 450, 455, 475, 476, 477
        ],
        "trash": [
            29, 30, 31, 32, 33, 37, 49, 53, 56, 58, 80, 82, 83, 106, 107, 108, 123, 124, 135
        ],
    },
    "prueba": {
        "cardboard": [
            3, 9, 14, 16, 23, 30, 43, 44, 45, 64, 65, 67, 87, 90, 103, 138, 139, 157, 162, 172,
            183, 184, 185, 186, 187, 210, 211, 212, 213, 231, 233, 246, 251, 254, 256, 258,
            270, 278, 280, 299, 314, 318, 324, 333, 338, 343, 345, 346, 353, 358, 367, 368,
            378, 384, 387, 388
        ],
        "glass": [
            2, 5, 10, 18, 20, 22, 31, 42, 45, 47, 50, 64, 68, 94, 99, 108, 109, 111, 112, 113,
            122, 130, 149, 153, 155, 179, 181, 199, 201, 207, 211, 213, 218, 224, 241, 243,
            246, 260, 266, 285, 290, 292, 293, 311, 312, 313, 314, 331, 334, 335, 347, 360,
            361, 376, 378, 383, 388, 403, 405, 408, 413, 415, 416, 429, 449, 450, 468, 470,
            487, 491, 492, 494
        ],
        "metal": [
            1, 3, 23, 30, 32, 37, 39, 41, 59, 63, 68, 84, 85, 86, 90, 111, 112, 113, 131, 133,
            135, 137, 139, 140, 168, 169, 179, 181, 185, 186, 202, 204, 213, 225, 226, 227,
            228, 251, 257, 261, 269, 273, 274, 278, 297, 311, 314, 318, 319, 339, 343, 345,
            359, 363, 380, 387, 406, 407, 408, 409
        ],
        "paper": [
            32, 34, 48, 50, 56, 59, 65, 70, 73, 92, 96, 97, 105, 111, 112, 131, 135, 143, 149,
            153, 158, 159, 181, 183, 188, 200, 202, 205, 209, 212, 215, 228, 239, 242, 246,
            247, 248, 268, 269, 270, 285, 297, 298, 299, 318, 319, 320, 327, 340, 341, 349,
            353, 361, 364, 386, 387, 388, 406, 407, 410, 413, 426, 430, 431, 436, 439, 443,
            445, 455, 481, 482, 496, 499, 500, 507, 519, 523, 526, 527, 528, 540, 561, 562,
            575, 583, 584, 586, 587
        ],
        "plastic": [
            16, 18, 23, 25, 29, 40, 45, 59, 61, 81, 83, 85, 89, 94, 111, 123, 129, 141, 144,
            146, 147, 164, 165, 191, 203, 210, 212, 229, 230, 250, 251, 252, 261, 262, 271,
            276, 277, 297, 298, 316, 317, 319, 338, 339, 342, 354, 357, 364, 370, 374, 379,
            381, 393, 395, 396, 400, 422, 428, 429, 445, 446, 458, 460, 466, 470, 473, 474
        ],
        "trash": [
            19, 23, 27, 28, 48, 50, 54, 76, 77, 79, 81, 97, 103, 105, 120, 121, 126, 127, 133,
            137
        ],
    },
}

# Tabla por clase registrada en la fase 2: [entrenamiento, validación, prueba].
EXPECTED_SPLIT = {"cardboard": [286, 61, 56], "glass": [352, 74, 72], "metal": [293, 56, 60], "paper": [423, 83, 88], "plastic": [346, 67, 67], "trash": [98, 19, 20]}

## 1. Partición del conjunto de datos
---

TrashNet no trae una partición oficial, así que el equipo la definió en las fases 1 y 2:

- **70 % / 15 % / 15 %** para entrenamiento, validación y prueba, **estratificada** por clase para conservar la proporción de `trash` en las tres particiones.
- **Agrupada:** las imágenes casi duplicadas (mismo objeto fotografiado dos veces, distancia pHash ≤ 10 dentro de la clase) quedan siempre en la misma partición. Así la validación y la prueba no premian la memorización.
- **Fijada archivo por archivo:** recalcularla con otra versión de scikit-learn cambiaría más de mil asignaciones, por eso se reconstruye desde las listas y se verifica con una huella SHA-256.

Cada partición tiene un único uso:

| Partición | Uso |
|---|---|
| Entrenamiento | Ajustar los pesos de los modelos y calcular los pesos por clase |
| Validación | Parada temprana, comparación de modelos y elección de hiperparámetros |
| Prueba | Solo la evaluación final de la fase 5, con las configuraciones ya congeladas |

In [ ]:
excluded = set(EXCLUIDAS)
assert excluded <= set(manifest_names), "Faltan archivos excluidos en el ZIP."
usable = pd.DataFrame({"ruta": [n for n in manifest_names if n not in excluded]})
usable["clase"] = usable["ruta"].str.split("/").str[0]
usable["numero"] = usable["ruta"].str.extract(r"(\d+)\.jpg$")[0].astype(int)
assignment = {(c, n): split for split, per_class in PARTICION_FASE2.items()
              for c, numbers in per_class.items() for n in numbers}
usable["particion"] = [assignment.get((c, n), "entrenamiento") for c, n in zip(usable["clase"], usable["numero"])]

split_table = pd.crosstab(usable["clase"], usable["particion"]).reindex(index=CLASSES, columns=SPLITS)
expected_table = pd.DataFrame(EXPECTED_SPLIT, index=SPLITS).T
PARTITION_FINGERPRINT = hashlib.sha256(
    "\n".join(f"{r}|{p}" for r, p in sorted(zip(usable["ruta"], usable["particion"]))).encode()).hexdigest()

partition_checks = {
    "2.527 imágenes en el ZIP, con los conteos por clase de la fuente":
        len(manifest_names) == 2527 and pd.Series([n.split("/")[0] for n in manifest_names]).value_counts().to_dict() == EXPECTED,
    "2.521 imágenes utilizables tras las 6 exclusiones": len(usable) == 2521,
    "Todos los archivos de validación y prueba existen": len(assignment) == int((usable["particion"] != "entrenamiento").sum()),
    "Tabla por clase igual a la de la fase 2": split_table.equals(expected_table.astype(split_table.dtypes.iloc[0])),
}
display(pd.DataFrame({"control": list(partition_checks), "resultado": ["OK" if v else "REVISAR" for v in partition_checks.values()]}))
assert all(partition_checks.values()), "La partición no coincide con la fase 2."

shown = split_table.copy()
shown["Total"] = shown.sum(axis=1)
shown.loc["Total"] = shown.sum()
display(shown.rename(index=LABEL_ES, columns=SPLIT_ES))
sizes = split_table.sum()
print("Proporción:", " / ".join(pct(sizes[s] / sizes.sum()) for s in SPLITS))
print("Huella SHA-256 de la asignación:", PARTITION_FINGERPRINT[:16], "…")

In [ ]:
# @title Gráfica de la partición por clase
fig, ax = plt.subplots(figsize=(10, 3.2))
left = np.zeros(len(CLASSES))
split_colors = {"entrenamiento": "#2a78d6", "validacion": "#eb6834", "prueba": "#1baf7a"}
for split in SPLITS:
    values = split_table[split].to_numpy()
    ax.barh([LABEL_ES[c] for c in CLASSES], values, left=left, color=split_colors[split], label=SPLIT_ES[split], height=0.62)
    left += values
ax.invert_yaxis()
ax.set_xlabel("Imágenes")
ax.legend(ncol=3, loc="lower right")
ax.set_title("Partición por clase: la proporción se conserva y `trash` está en las tres particiones", loc="left")
fig.tight_layout()
plt.show()

## 2. Diseño de la configuración experimental
---

### 2.1 Tipo de problema y pregunta experimental

- **Tarea:** clasificación supervisada **multiclase**. La entrada es una fotografía RGB y la salida, una de 6 clases mutuamente excluyentes. La capa final es *softmax* de 6 neuronas y la pérdida, la entropía cruzada categórica.
- **Pregunta experimental:** ¿la **transferencia de aprendizaje** desde ImageNet mejora la clasificación frente a una **CNN entrenada desde cero**, con los mismos datos, la misma entrada y el mismo protocolo?
- **Hipótesis:** con 1.798 imágenes de entrenamiento, las representaciones preentrenadas deberían generalizar mejor que las aprendidas desde cero, que tienden a sobreajustar.

### 2.2 Métricas

| Métrica | Papel | Por qué |
|---|---|---|
| **F1 macro (validación)** | **Principal**: parada temprana, selección de modelo | Promedia el F1 de las 6 clases por igual: `trash` (5,5 % de los datos) pesa lo mismo que `paper` |
| Recall mínimo por clase | Secundaria | Detecta una clase abandonada aunque el promedio sea bueno |
| Accuracy | Secundaria | Comparable con la literatura; engañosa sola (la mayoritaria logra 0,23) |
| Matriz de confusión | Análisis | Muestra qué materiales se confunden (en la fase 2: `glass` → `plastic`) |
| Tiempo por época y parámetros | Costo | El escenario de uso es una estación educativa: importa que el modelo sea liviano |

**Metas de la fase 1:** F1 macro ≥ 0,80 y recall ≥ 0,60 en cada clase, en prueba. **Mínimo para justificar la complejidad:** superar con claridad el F1 macro de 0,528 de los descriptores globales (fase 2).

### 2.3 Configuraciones a comparar

| Id | Modelo | Escala de entrada | Optimizador | Tasa de aprendizaje | Épocas máx. | Paciencia | Fase |
|---|---|---|---|---|---|---|---|
| **A** | CNN pequeña desde cero | [0, 1] | Adam | 1e-3 | 40 | 6 | 4 |
| **B** | MobileNetV2 congelada + cabeza nueva | [−1, 1] | Adam | 1e-3 | 20 | 5 | 4 |
| **B-FT** | B + ajuste fino de las últimas 30 capas | [−1, 1] | Adam | 1e-5 | 10 | 3 | 4 (si B lo justifica) |

**Elementos comunes a todas las configuraciones:**
- Entrada de **160 × 160 × 3** y lote de **32** imágenes.
- Los **mismos aumentos**, solo en entrenamiento: reflexión, rotación ± 14°, zoom ± 10 %, brillo, contraste y ganancia por canal.
- **Pesos por clase** de entrenamiento.
- **Parada temprana** sobre el F1 macro de validación, con restauración de los mejores pesos, y **checkpoint** de esos pesos.
- Semilla 42.

**Presupuesto:** máximo 6 entrenamientos completos en la fase 4. El ensayo corto de la sección 4.4 mide el tiempo real por época para confirmarlo.

### 2.4 Comparaciones adicionales (fase 4)

1. **Con y sin pesos por clase** sobre el mejor modelo: ¿mejoran el recall de `trash` sin sacrificar el F1 macro?
2. **Con y sin ganancia por canal:** ¿reduce la dependencia del tono del fondo, que en la fase 2 predecía la clase con F1 0,403?
3. **Prueba del atajo de color:** evaluar en validación con **normalización de color *gray world*** (sección 4.5). Si el F1 cae mucho, el modelo depende del tinte de cada sesión de fotos y no solo del material.

### 2.5 Reglas que evitan conclusiones engañosas

- La partición de **prueba no se usa** en ninguna decisión. La función de datos se niega a entregarla salvo autorización explícita en la fase 5.
- Todas las configuraciones comparten partición, semilla, aumentos y pesos por clase: la única diferencia es el modelo.
- Los resultados de la sección 4.4 vienen de **dos épocas de prueba**. Sirven para verificar la implementación y medir tiempos, no para comparar modelos.

## 3. Selección y diseño de los modelos
---

### 3.1 Por qué redes convolucionales

Es un problema de **análisis de imagen**: el material se reconoce por textura, brillo, transparencia y forma. Las capas convolucionales aprenden esos patrones locales con pocos parámetros y los reconocen en cualquier posición de la foto. Un modelo denso sobre píxeles crudos necesitaría millones de pesos para la misma entrada.

### 3.2 Modelo A · CNN pequeña desde cero

| Bloque | Capas | Salida |
|---|---|---|
| Entrada | Imagen 160 × 160 × 3 en [0, 1] | 160 × 160 × 3 |
| 1 | Conv 3×3 (32) → BatchNorm → ReLU → MaxPool 2×2 | 80 × 80 × 32 |
| 2 | Conv 3×3 (64) → BatchNorm → ReLU → MaxPool 2×2 | 40 × 40 × 64 |
| 3 | Conv 3×3 (128) → BatchNorm → ReLU → MaxPool 2×2 | 20 × 20 × 128 |
| 4 | Conv 3×3 (256) → BatchNorm → ReLU → MaxPool 2×2 | 10 × 10 × 256 |
| Cabeza | Global average pooling → Dropout 0,3 → Dense 6 *softmax* | 6 |

**Por qué este diseño:**
- **Profundidad:** cuatro bloques duplican los filtros mientras reducen la resolución. Así el modelo pasa de bordes y colores a texturas y partes de objetos, como en la jerarquía de la unidad 3.
- **Tamaño:** se mantiene **pequeño a propósito** (≈ 0,4 millones de parámetros) porque solo hay 1.798 imágenes de entrenamiento.
- ***Global average pooling*** en lugar de aplanar: evita una capa densa de millones de pesos y reduce el sobreajuste.
- **BatchNorm** estabiliza el entrenamiento desde cero y ***dropout*** regulariza la cabeza. BatchNorm usa *momentum* 0,9 en lugar de 0,99: con solo 57 lotes por época, sus estadísticas de inferencia se ajustan en pocas épocas y la validación refleja antes lo aprendido.

Es la **línea base de aprendizaje profundo** de la pregunta experimental.

### 3.3 Modelo B · MobileNetV2 con transferencia de aprendizaje

| Parte | Detalle |
|---|---|
| Extractor | MobileNetV2 (α = 1,0) con pesos de **ImageNet**, sin su clasificador original (`include_top=False`), entrada 160 × 160 × 3 en [−1, 1] |
| Estado inicial | **Congelado**: solo se entrena la cabeza nueva |
| Cabeza nueva | Global average pooling → Dropout 0,2 → Dense 6 *softmax* |
| Variante B-FT | Descongela las últimas 30 capas, con BatchNorm en modo inferencia y tasa de aprendizaje 1e-5 |

**Por qué MobileNetV2:**
- **Representaciones preentrenadas:** ImageNet incluye botellas, latas, cajas y papeles. Sus filtros ya distinguen texturas y reflejos, justo lo que separa los materiales.
- **Eficiencia:** las convoluciones separables en profundidad dan unos 2,3 millones de parámetros, frente a más de 20 millones de un ResNet-50. Encaja con una estación educativa que podría funcionar en un equipo modesto.
- **Entrada compatible:** 160 × 160 es una de las resoluciones con pesos publicados, la misma que dejó lista el pipeline de la fase 2.
- **Protocolo de la unidad 3:** primero **transfer learning** con el extractor congelado (*warming up* de la cabeza) y después **ajuste fino** con una tasa de aprendizaje pequeña, para no destruir los pesos preentrenados.

### 3.4 Cómo se ajusta el diseño a los datos

| Hallazgo de la fase 2 | Respuesta en el diseño |
|---|---|
| Solo 1.798 imágenes de entrenamiento | CNN pequeña con GAP y *dropout*; transferencia de aprendizaje como apuesta principal |
| Desbalance 4,32 : 1, concentrado en `trash` | Pesos por clase y F1 macro como métrica de selección |
| El tono del fondo predice la clase (F1 0,403) | Aumento de ganancia por canal y prueba de *gray world* en validación |
| Fotos 4:3 con objetos que llegan a los bordes | Entrada cuadrada con relleno, sin recortar el objeto |
| Atípicos de exposición y nitidez | Aumentos moderados de brillo y contraste |

## 4. Implementación de los modelos
---

### 4.1 Pipeline de entrada

El mismo pipeline de la fase 2:
- caché en memoria a 160 × 120;
- relleno a 160 × 160;
- aumentos por lote solo en entrenamiento;
- escala propia de cada modelo, aplicada una sola vez.

Se agregan dos cosas:
1. **Una protección para la partición de prueba**, que no se entrega sin autorización explícita.
2. **La opción de normalizar el color con *gray world***, que iguala la media de los tres canales de cada imagen. Sirve para la prueba del atajo de color.

In [ ]:
CONTENT_HW = (120, 160)                       # alto, ancho: conserva 4:3
IMG_SIZE = 160
PAD_TOP = (IMG_SIZE - CONTENT_HW[0]) // 2     # 20 filas de relleno arriba y abajo
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_resized(relative):
    """Decodifica a escala 1/2 (DCT del JPEG) y reduce con Lanczos a 160 × 120 en uint8."""
    with Image.open(DATA_DIR / relative) as image:
        image.draft("RGB", (CONTENT_HW[1], CONTENT_HW[0]))
        rgb = image.convert("RGB")
        return np.asarray(rgb.resize((CONTENT_HW[1], CONTENT_HW[0]), Image.Resampling.LANCZOS), dtype=np.uint8)


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    X_all = np.stack(list(pool.map(load_resized, usable["ruta"])))
y_all = usable["clase"].map(CLASS_INDEX).to_numpy(np.int32)
CACHE = {split: (X_all[(usable["particion"] == split).to_numpy()], y_all[(usable["particion"] == split).to_numpy()])
         for split in SPLITS}
print(f"Caché: {X_all.nbytes / 1e6:.0f} MB en {time.perf_counter() - t0:.1f} s | "
      + " · ".join(f"{SPLIT_ES[s]} {len(CACHE[s][1])}" for s in SPLITS))

AUGMENTATION = {"reflexion": "horizontal", "rotacion": 0.04, "zoom": 0.10, "brillo": 0.10,
                "contraste": 0.15, "ganancia_canal": 0.06}
augmenter = keras.Sequential([
    keras.layers.RandomFlip(AUGMENTATION["reflexion"], seed=SEED),
    keras.layers.RandomRotation(AUGMENTATION["rotacion"], fill_mode="reflect", seed=SEED + 1),
    keras.layers.RandomZoom(AUGMENTATION["zoom"], fill_mode="reflect", seed=SEED + 2),
    keras.layers.RandomBrightness(AUGMENTATION["brillo"], value_range=(0, 255), seed=SEED + 3),
    keras.layers.RandomContrast(AUGMENTATION["contraste"], seed=SEED + 4),
], name="aumentos_entrenamiento")


def augment_batch(images):
    """Aumentos geométricos y fotométricos sobre un lote float32 en [0, 255], antes del relleno."""
    images = augmenter(images, training=True)
    g = AUGMENTATION["ganancia_canal"]
    gains = tf.random.uniform((tf.shape(images)[0], 1, 1, 3), 1 - g, 1 + g)
    return tf.clip_by_value(images * gains, 0.0, 255.0)


def gray_world(images):
    """Normaliza el color: iguala la media de los tres canales de cada imagen (elimina el tinte global)."""
    means = tf.reduce_mean(images, axis=[1, 2], keepdims=True)
    gray = tf.reduce_mean(means, axis=-1, keepdims=True)
    return tf.clip_by_value(images * gray / (means + 1e-6), 0.0, 255.0)


def pad_and_scale(images, scale):
    """Relleno a 160 × 160 y escala propia de cada modelo."""
    images = tf.image.pad_to_bounding_box(images, PAD_TOP, 0, IMG_SIZE, IMG_SIZE)
    if scale == "cnn":
        return images / 255.0
    if scale == "mobilenet":
        return images / 127.5 - 1.0          # idéntico a mobilenet_v2.preprocess_input
    raise ValueError("scale debe ser 'cnn' o 'mobilenet'")


TEST_ACCESS = {"autorizado": False, "solicitudes": 0}


def make_dataset(split, scale, batch_size=BATCH_SIZE, augment=None, shuffle=None, color_norm=False):
    """Dataset para Keras. Aumentos y barajado solo en entrenamiento; prueba bloqueada hasta la fase 5."""
    if split == "prueba":
        TEST_ACCESS["solicitudes"] += 1
        if not TEST_ACCESS["autorizado"]:
            raise PermissionError("La partición de prueba está reservada para la evaluación final (fase 5).")
    augment = (split == "entrenamiento") if augment is None else augment
    shuffle = (split == "entrenamiento") if shuffle is None else shuffle
    if augment and split != "entrenamiento":
        raise ValueError("Los aumentos solo se aplican al entrenamiento.")
    images, labels = CACHE[split]
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(len(labels), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)

    def transform(x, y):
        x = tf.cast(x, tf.float32)
        if augment:
            x = augment_batch(x)
        if color_norm:
            x = gray_world(x)
        return pad_and_scale(x, scale), y

    return ds.map(transform, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


CLASS_WEIGHT = {i: float(w) for i, w in enumerate(
    compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=CACHE["entrenamiento"][1]))}
print("Pesos por clase:", {LABEL_ES[CLASSES[i]]: round(w, 3) for i, w in CLASS_WEIGHT.items()})

In [ ]:
# @title Verificación visual de la normalización de color
sample_idx = [int(np.where(CACHE["validacion"][1] == CLASS_INDEX[c])[0][0]) for c in CLASSES]
sample = tf.constant(CACHE["validacion"][0][sample_idx], tf.float32)
normalized = gray_world(sample)
fig, axes = plt.subplots(2, len(CLASSES), figsize=(13, 3.9))
for col, c in enumerate(CLASSES):
    for row, (images, name) in enumerate([(sample, "original"), (normalized, "gray world")]):
        ax = axes[row, col]
        ax.imshow(images[col].numpy().astype("uint8"))
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if row == 0:
            ax.set_title(LABEL_ES[c], fontsize=10)
        if col == 0:
            ax.set_ylabel(name, fontsize=9, color=INK_2)
fig.suptitle("Normalización gray world: se elimina el tinte global y se conservan forma y textura",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout()
plt.show()

### 4.2 Construcción de los modelos

In [ ]:
def build_cnn(n_classes=len(CLASSES), filters=(32, 64, 128, 256), dropout=0.3):
    """Modelo A: CNN pequeña entrenada desde cero."""
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="imagen")
    x = inputs
    for i, f in enumerate(filters, start=1):
        x = keras.layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"conv{i}")(x)
        x = keras.layers.BatchNormalization(momentum=0.9, name=f"bn{i}")(x)
        x = keras.layers.ReLU(name=f"relu{i}")(x)
        x = keras.layers.MaxPooling2D(2, name=f"pool{i}")(x)
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(dropout, name="dropout")(x)
    outputs = keras.layers.Dense(n_classes, activation="softmax", name="clases")(x)
    return keras.Model(inputs, outputs, name="A_cnn_pequena")


def build_mobilenet(n_classes=len(CLASSES), dropout=0.2, weights="imagenet"):
    """Modelo B: MobileNetV2 de ImageNet congelada con una cabeza nueva."""
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                          weights=weights, alpha=1.0)
    base.trainable = False
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="imagen")
    x = base(inputs, training=False)          # BatchNorm del extractor siempre en modo inferencia
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(dropout, name="dropout")(x)
    outputs = keras.layers.Dense(n_classes, activation="softmax", name="clases")(x)
    return keras.Model(inputs, outputs, name="B_mobilenetv2_transfer")


def unfreeze_top(model, n_layers=30):
    """Variante B-FT: descongela las últimas capas del extractor; las BatchNorm siguen congeladas."""
    base = next(layer for layer in model.layers if isinstance(layer, keras.Model))
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
    return model


def count_params(model):
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    total = int(sum(np.prod(w.shape) for w in model.weights))
    return trainable, total


keras.utils.set_random_seed(SEED)
model_a = build_cnn()
model_b = build_mobilenet()
model_a.summary(line_length=96)
model_b.summary(line_length=96)

In [ ]:
# @title Parámetros por configuración
model_ft = unfreeze_top(build_mobilenet())
param_rows = []
for key, model, desc in [("A", model_a, "CNN pequeña desde cero"),
                         ("B", model_b, "MobileNetV2 congelada + cabeza"),
                         ("B-FT", model_ft, "MobileNetV2, últimas 30 capas entrenables")]:
    trainable, total = count_params(model)
    param_rows.append({"id": key, "modelo": desc, "parámetros totales": miles(total),
                       "entrenables": miles(trainable), "% entrenable": pct(trainable / total)})
PARAMS = pd.DataFrame(param_rows).set_index("id")
display(PARAMS)

### 4.3 Compilación y verificaciones antes de entrenar

Cada modelo se compila con:
- **Adam**;
- **entropía cruzada categórica dispersa**, porque las etiquetas son enteros de 0 a 5;
- **accuracy** como métrica de Keras.

El **F1 macro de validación** se calcula al final de cada época con un *callback* propio. La parada temprana y el checkpoint lo usan como criterio.

Antes de gastar tiempo de GPU se hacen tres verificaciones estándar:
1. **Forma y probabilidades:** cada salida tiene 6 valores que suman 1.
2. **Pérdida inicial:** un modelo sin entrenar debe predecir casi uniforme, así que la pérdida debe estar cerca de ln 6 ≈ 1,79.
3. **Sobreajuste de un lote:** con 60 pasos sobre un mismo lote, la pérdida debe caer cerca de cero. Si no lo hace, hay un error en el modelo, las etiquetas o los gradientes.

In [ ]:
EXPERIMENTS = {
    "A":    {"modelo": "CNN pequeña desde cero", "builder": build_cnn, "escala": "cnn",
             "lr": 1e-3, "epocas_max": 40, "paciencia": 6},
    "B":    {"modelo": "MobileNetV2 congelada + cabeza", "builder": build_mobilenet, "escala": "mobilenet",
             "lr": 1e-3, "epocas_max": 20, "paciencia": 5},
    "B-FT": {"modelo": "MobileNetV2 + ajuste fino (30 capas)", "builder": lambda: unfreeze_top(build_mobilenet()),
             "escala": "mobilenet", "lr": 1e-5, "epocas_max": 10, "paciencia": 3},
}


def compile_model(model, learning_rate):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
    return model


class MacroF1(keras.callbacks.Callback):
    """Agrega a los registros de cada época el F1 macro y el recall mínimo en validación."""

    def __init__(self, scale):
        super().__init__()
        self.dataset = make_dataset("validacion", scale)
        self.labels = CACHE["validacion"][1]

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.dataset, verbose=0).argmax(axis=1)
        logs["val_f1_macro"] = f1_score(self.labels, pred, average="macro")
        logs["val_recall_min"] = recall_score(self.labels, pred, average=None,
                                              labels=range(len(CLASSES)), zero_division=0).min()


def make_callbacks(exp_id):
    """F1 macro → parada temprana y checkpoint sobre ese F1 → registro CSV. El orden importa."""
    cfg = EXPERIMENTS[exp_id]
    safe = exp_id.replace("-", "_")
    return [
        MacroF1(cfg["escala"]),
        keras.callbacks.EarlyStopping(monitor="val_f1_macro", mode="max", patience=cfg["paciencia"],
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(str(MODEL_DIR / f"{safe}.weights.h5"), monitor="val_f1_macro",
                                        mode="max", save_best_only=True, save_weights_only=True),
        keras.callbacks.CSVLogger(str(REPORT_DIR / f"historial_{safe}.csv")),
    ]


for exp_id in EXPERIMENTS:
    names = [type(cb).__name__ for cb in make_callbacks(exp_id)]
    print(f"{exp_id}: lr={EXPERIMENTS[exp_id]['lr']:g} · épocas máx.={EXPERIMENTS[exp_id]['epocas_max']} · "
          f"paciencia={EXPERIMENTS[exp_id]['paciencia']} · callbacks={names}")

In [ ]:
# @title Verificaciones: forma, pérdida inicial y sobreajuste de un lote
smoke = []
for exp_id in ("A", "B"):
    keras.utils.set_random_seed(SEED)
    cfg = EXPERIMENTS[exp_id]
    model = compile_model(cfg["builder"](), cfg["lr"])
    xb, yb = next(iter(make_dataset("entrenamiento", cfg["escala"], augment=False, shuffle=False)))
    probs = model(xb, training=False).numpy()
    initial_loss = float(keras.losses.SparseCategoricalCrossentropy()(yb, probs))
    t0 = time.perf_counter()
    for _ in range(60):
        logs = model.train_on_batch(xb, yb, return_dict=True)
    smoke.append({"id": exp_id, "forma de salida": str(probs.shape),
                  "suma de probabilidades": f"{probs.sum(axis=1).min():.4f} – {probs.sum(axis=1).max():.4f}",
                  "pérdida inicial": round(initial_loss, 3), "referencia ln 6": round(float(np.log(6)), 3),
                  "pérdida tras 60 pasos en un lote": round(float(logs["loss"]), 4),
                  "accuracy en ese lote": round(float(logs["accuracy"]), 3),
                  "segundos": round(time.perf_counter() - t0, 1)})
SMOKE = pd.DataFrame(smoke).set_index("id")
display(SMOKE)

### 4.4 Ensayo corto de entrenamiento

Cada modelo se entrena **dos épocas** con el pipeline completo:
- aumentos;
- pesos por clase;
- *callback* de F1 macro.

La primera época incluye la compilación del grafo, así que **el tiempo de referencia es el de la segunda**. Con ese tiempo se estima el costo de los entrenamientos completos de la fase 4. **Dos épocas no bastan para comparar modelos.**

In [ ]:
# @title Ensayo corto: dos épocas por modelo con medición de tiempo
trial_rows, trial_histories = [], {}
for exp_id in ("A", "B"):
    keras.utils.set_random_seed(SEED)
    cfg = EXPERIMENTS[exp_id]
    model = compile_model(cfg["builder"](), cfg["lr"])
    timer = []
    timing = keras.callbacks.LambdaCallback(on_epoch_begin=lambda e, l: timer.append(time.perf_counter()),
                                            on_epoch_end=lambda e, l: timer.append(time.perf_counter()))
    history = model.fit(make_dataset("entrenamiento", cfg["escala"]),
                        validation_data=make_dataset("validacion", cfg["escala"]),
                        epochs=2, class_weight=CLASS_WEIGHT, verbose=2,
                        callbacks=[timing, MacroF1(cfg["escala"])])
    epoch_seconds = [timer[2 * k + 1] - timer[2 * k] for k in range(2)]
    trial_histories[exp_id] = pd.DataFrame(history.history)
    last = history.history
    trial_rows.append({"id": exp_id, "modelo": cfg["modelo"], "dispositivo": DEVICE,
                       "s época 1": round(epoch_seconds[0], 1), "s época 2": round(epoch_seconds[1], 1),
                       "pérdida entren. (ép. 2)": round(last["loss"][-1], 3),
                       "pérdida valid. (ép. 2)": round(last["val_loss"][-1], 3),
                       "F1 macro valid. (ép. 2)": round(last["val_f1_macro"][-1], 3),
                       "estimado entrenamiento completo (min)": round(epoch_seconds[1] * cfg["epocas_max"] / 60, 1)})
TRIAL = pd.DataFrame(trial_rows).set_index("id")
display(TRIAL)

### 4.5 Línea base de transferencia y prueba del atajo de color

Antes de entrenar la cabeza del modelo B por descenso de gradiente, conviene saber cuánta información ya está en las representaciones de ImageNet. Para eso:
1. Se pasa cada imagen, **sin aumentos**, por MobileNetV2 congelada con *global average pooling*, lo que da un vector de 1.280 características.
2. Sobre esos vectores se ajusta una **regresión logística** con pesos balanceados.

Es la referencia más exigente para la fase 4: si el modelo B entrenado no la supera, la cabeza o los aumentos están fallando.

Con el mismo clasificador se evalúa la validación **normalizada con *gray world***. Si el F1 cae mucho, las predicciones dependen del tinte de cada sesión de fotos.

*Gray world* también altera el color propio de algunos materiales (por ejemplo, el marrón del cartón), así que la caída es una **cota superior** de la dependencia del fondo.

In [ ]:
# @title Línea base de transferencia y prueba del atajo de color
extractor = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                           weights="imagenet", pooling="avg")
extractor.trainable = False


def features(split, color_norm=False):
    ds = make_dataset(split, "mobilenet", augment=False, shuffle=False, color_norm=color_norm)
    return extractor.predict(ds, verbose=0)


t0 = time.perf_counter()
F_train, F_val = features("entrenamiento"), features("validacion")
F_val_gw = features("validacion", color_norm=True)
y_train, y_val = CACHE["entrenamiento"][1], CACHE["validacion"][1]
transfer_baseline = make_pipeline(StandardScaler(),
                                  LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5))
transfer_baseline.fit(F_train, y_train)
feature_seconds = time.perf_counter() - t0


def summarize(name, y_true, y_pred):
    recalls = recall_score(y_true, y_pred, average=None, labels=range(len(CLASSES)), zero_division=0)
    return {"referencia": name, "accuracy": accuracy_score(y_true, y_pred),
            "F1 macro": f1_score(y_true, y_pred, average="macro"), "recall mínimo": recalls.min(),
            "clase con menor recall": CLASSES[int(recalls.argmin())], "recall trash": recalls[CLASS_INDEX["trash"]]}


pred_val = transfer_baseline.predict(F_val)
pred_val_gw = transfer_baseline.predict(F_val_gw)
fase2 = [{"referencia": "Mayoritaria (fase 2)", "accuracy": 0.231, "F1 macro": 0.062},
         {"referencia": "Solo fondo (fase 2)", "accuracy": 0.433, "F1 macro": 0.403},
         {"referencia": "Descriptores globales (fase 2)", "accuracy": 0.547, "F1 macro": 0.528}]
BASELINES = pd.DataFrame(fase2 + [summarize("MobileNetV2 congelada + regresión logística", y_val, pred_val),
                                  summarize("La misma, validación con gray world", y_val, pred_val_gw)]).set_index("referencia")
display(BASELINES.style.format(precision=3, na_rep="—"))
print(f"Extracción de características y ajuste: {feature_seconds:.1f} s en {DEVICE}")

In [ ]:
# @title Matriz de confusión y lectura de la línea base
cm = confusion_matrix(y_val, pred_val, labels=range(len(CLASSES)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(cm_norm, cmap=SEQUENTIAL, vmin=0, vmax=1)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                color="white" if cm_norm[i, j] > 0.6 else INK)
ax.set_xticks(range(len(CLASSES)), [LABEL_ES[c] for c in CLASSES], rotation=35, ha="right")
ax.set_yticks(range(len(CLASSES)), [LABEL_ES[c] for c in CLASSES])
ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real")
ax.set_title("Línea base de transferencia · validación", loc="left")
fig.colorbar(im, ax=ax, fraction=0.046, label="proporción de la fila")
fig.tight_layout()
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
b = BASELINES.loc["MobileNetV2 congelada + regresión logística"]
g = BASELINES.loc["La misma, validación con gray world"]
display(Markdown(
    f"**Lectura.** Las representaciones de ImageNet, sin entrenar la red, llevan el F1 macro de validación a "
    f"**{dec(b['F1 macro'], 3)}**, frente a {dec(0.528, 3)} de los descriptores globales de la fase 2. La clase con menor "
    f"recall es `{b['clase con menor recall']}` ({dec(b['recall mínimo'], 3)}) y la confusión más frecuente es "
    f"`{CLASSES[i]}` → `{CLASSES[j]}` ({cm[i, j]} imágenes). Con *gray world* el F1 macro pasa a "
    f"{dec(g['F1 macro'], 3)} (diferencia de {dec(g['F1 macro'] - b['F1 macro'], 3)}): "
    + ("no cae, así que estas representaciones no dependen del tinte de cada sesión para acertar."
       if g["F1 macro"] >= b["F1 macro"] else
       "la caída es pequeña, así que estas representaciones dependen poco del tinte de cada sesión."
       if g["F1 macro"] >= b["F1 macro"] - 0.05 else
       "la caída es apreciable, señal de que parte de la información viene del tinte de cada sesión; "
       "la comparación con y sin ganancia por canal de la fase 4 queda plenamente justificada.")
))

## 5. Controles automáticos y trazabilidad
---

In [ ]:
# @title Controles automáticos
trainable_b, _ = count_params(model_b)
head_params = 1280 * len(CLASSES) + len(CLASSES)
checks = {
    "Partición idéntica a la fase 2 (tabla por clase)": all(partition_checks.values()),
    "Las seis clases están en las tres particiones": bool(split_table.gt(0).all().all()),
    "Salidas de forma (lote, 6) con probabilidades que suman 1":
        all(s.startswith("(32, 6)") for s in SMOKE["forma de salida"]),
    "Pérdida inicial a menos de 0,5 de ln 6": bool((SMOKE["pérdida inicial"] - np.log(6)).abs().lt(0.5).all()),
    "Cada modelo sobreajusta un lote (pérdida < 0,2)": bool(SMOKE["pérdida tras 60 pasos en un lote"].lt(0.2).all()),
    "Modelo B: solo la cabeza es entrenable (7.686 parámetros)": trainable_b == head_params,
    "B-FT entrena más parámetros que B y menos que el total":
        count_params(model_b)[0] < count_params(model_ft)[0] < count_params(model_ft)[1],
    "Escala MobileNetV2 igual a preprocess_input": bool(np.allclose(
        pad_and_scale(tf.constant(CACHE["validacion"][0][:4], tf.float32), "mobilenet").numpy()[:, PAD_TOP:PAD_TOP + 120],
        keras.applications.mobilenet_v2.preprocess_input(CACHE["validacion"][0][:4].astype("float32")), atol=1e-5)),
    "Pesos por clase igualan el aporte de cada clase": bool(np.allclose(
        np.bincount(CACHE["entrenamiento"][1]) * np.array(list(CLASS_WEIGHT.values())),
        len(CACHE["entrenamiento"][1]) / len(CLASSES))),
    "La partición de prueba no se solicitó en ningún momento": TEST_ACCESS["solicitudes"] == 0,
}
CHECKS = pd.DataFrame({"control": list(checks), "resultado": ["OK" if v else "REVISAR" for v in checks.values()]})
display(CHECKS)
assert all(checks.values()), "Un control falló: revisar antes de la fase 4."

In [ ]:
# @title Archivos de trazabilidad
config = {
    "ejecutado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "entorno": {"dispositivo": DEVICE, "versiones": versions, "semilla": SEED},
    "datos": {"fuente": source["nombre"], "huella_particion": PARTITION_FINGERPRINT,
              "tamanos": {s: int(sizes[s]) for s in SPLITS}},
    "entrada": {"forma": [IMG_SIZE, IMG_SIZE, 3], "contenido": list(CONTENT_HW), "lote": BATCH_SIZE,
                "aumentos": AUGMENTATION},
    "pesos_por_clase": {CLASSES[i]: round(w, 4) for i, w in CLASS_WEIGHT.items()},
    "experimentos": {k: {kk: vv for kk, vv in v.items() if kk != "builder"} for k, v in EXPERIMENTS.items()},
    "parametros": PARAMS.to_dict(orient="index"),
    "ensayo_corto": TRIAL.to_dict(orient="index"),
    "lineas_base": BASELINES.round(4).astype(object).where(BASELINES.notna(), None).to_dict(orient="index"),
}
(REPORT_DIR / "config_fase3.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
usable[["ruta", "clase", "particion"]].to_csv(REPORT_DIR / "particion_fase3.csv", index=False)
for model in (model_a, model_b):
    lines = []
    model.summary(print_fn=lambda line, **kwargs: lines.append(line), line_length=96)
    (REPORT_DIR / f"arquitectura_{model.name}.txt").write_text("\n".join(lines), encoding="utf-8")
print("Archivos generados en residuos_fase3/reportes:")
for path in sorted(REPORT_DIR.iterdir()):
    print(f"• {path.name} ({path.stat().st_size / 1024:.0f} KB)")

## 6. Conclusiones y próximos pasos
---

In [ ]:
# @title Conclusiones
b = BASELINES.loc["MobileNetV2 congelada + regresión logística"]
g = BASELINES.loc["La misma, validación con gray world"]
ta, tb = TRIAL.loc["A"], TRIAL.loc["B"]
display(Markdown(
    f"1. **Partición:** se reutiliza la de la fase 2, verificada archivo por archivo "
    f"({miles(sizes['entrenamiento'])} / {miles(sizes['validacion'])} / {miles(sizes['prueba'])}). La prueba permanece bloqueada.\n"
    f"2. **Diseño experimental:** dos modelos comparables con la misma entrada, los mismos aumentos, pesos por clase y "
    f"parada temprana sobre el **F1 macro de validación**, más una variante de ajuste fino y dos comparaciones acotadas.\n"
    f"3. **Modelos:** la CNN pequeña tiene {PARAMS.loc['A', 'parámetros totales']} parámetros, todos entrenables; "
    f"MobileNetV2 tiene {PARAMS.loc['B', 'parámetros totales']}, de los que solo {PARAMS.loc['B', 'entrenables']} "
    f"(la cabeza) se entrenan en la configuración B.\n"
    f"4. **Implementación verificada:** salidas de 6 probabilidades, pérdida inicial cercana a ln 6 y sobreajuste de un "
    f"lote en ambos modelos; {int((CHECKS['resultado'] == 'OK').sum())} de {len(CHECKS)} controles en OK.\n"
    f"5. **Costo medido en {DEVICE}:** {dec(ta['s época 2'], 1)} s por época para A y {dec(tb['s época 2'], 1)} s para B. "
    f"Los entrenamientos completos de la fase 4 tomarían unos {dec(ta['estimado entrenamiento completo (min)'], 1)} y "
    f"{dec(tb['estimado entrenamiento completo (min)'], 1)} minutos como máximo, dentro del presupuesto de 6 ejecuciones.\n"
    f"6. **Referencia de transferencia:** MobileNetV2 congelada con una regresión logística ya alcanza F1 macro "
    f"**{dec(b['F1 macro'], 3)}** en validación (frente a 0,528 de la fase 2). Con *gray world* queda en {dec(g['F1 macro'], 3)}."
))

### Próximos pasos (fase 4 · Entrenamiento y validación)

- Entrenar **A** y **B** completos con parada temprana sobre el F1 macro de validación y guardar sus curvas.
- Si B supera claramente a A, aplicar el **ajuste fino B-FT** sobre los mejores pesos de B.
- Correr las dos comparaciones acotadas sobre el mejor modelo: **con y sin pesos por clase** y **con y sin ganancia por canal**.
- Repetir la prueba de ***gray world*** con el modelo elegido y revisar la matriz de confusión de validación (`glass` frente a `plastic`).
- Congelar la configuración final para evaluarla una sola vez en prueba en la fase 5.

## 7. Referencias
---

1. Thung, G. y Yang, M. (2016). *TrashNet: repositorio y descripción del conjunto de datos*. [GitHub](https://github.com/garythung/trashnet).
2. Sandler, M., Howard, A., Zhu, M., Zhmoginov, A. y Chen, L.-C. (2018). MobileNetV2: Inverted Residuals and Linear Bottlenecks. *CVPR*.
3. Ioffe, S. y Szegedy, C. (2015). Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift. *ICML*.
4. Lin, M., Chen, Q. y Yan, S. (2014). Network In Network (*global average pooling*). *ICLR*.
5. Srivastava, N. et al. (2014). Dropout: A Simple Way to Prevent Neural Networks from Overfitting. *JMLR*, 15, 1929–1958.
6. Buchsbaum, G. (1980). A spatial processor model for object colour perception (*gray world*). *Journal of the Franklin Institute*, 310(1), 1–26.
7. Keras. *Transfer learning & fine-tuning*. [Guía oficial](https://keras.io/guides/transfer_learning/).
8. Keras. *MobileNetV2*. [Documentación](https://keras.io/api/applications/mobilenet/).
9. Material del curso: unidad 3, *Redes neuronales convolucionales* (transfer learning y ajuste fino), e *Instrucciones Proyecto aplicado – Fase 3*.
10. Notebooks de las fases 1 y 2 del equipo: *Entendimiento del negocio y carga de datos* y *Entendimiento y preparación de los datos*.